# Nigerian Fraud (419) preparation for Dataset 1

This notebook prepares `419_nigerian.txt` for the v2 Core human-side gathering stage.

The input is a raw mbox-format file containing ~3 978 advance-fee scam emails (2002 era).
The pipeline parses the mbox with Python's `mailbox` module, extracts the text body,
cleans and normalizes it, filters to English, deduplicates, and assigns Core schema fields.

Per `v2/docs/dataset_design_final.md` §5.6:
- `scenario_family = advance_fee_scam_email`
- `channel = email`, `fraudness = fraud`
- All records included after cleaning and quality filtering (no Core policy exclusion)
- `time_band = legacy` (entire corpus is 2002-era)

No subtype annotation — §5.6 does not require it as a preparation step.
The subtypes listed there (inheritance, diplomatic package, etc.) are prior diagnostic context only.

Pipeline:
- parse mbox: extract headers (Subject, From, Date) and text body,
- handle MIME multipart, quoted-printable, base64, and non-standard charsets,
- strip HTML tags from html-only parts,
- normalize text (NFKC, whitespace, broken encoding),
- URL masking (two-pass),
- English-like language filter,
- outlier cap: token_length ≤ 5 000 (removes parsing artifacts; max observed 26k tokens vs p90=758),
- deduplication by masked text,
- quality gate (char_length ≥ 50),
- assign all Core schema fields via `v2/config.py` for `length_bin`,
- export to `v2/data/interim/gathered/nigerian_fraud_prepared.jsonl`.

**Audit fix (Apr-2026):** added outlier cap; `length_bin` now imported from `v2/config.py`
(thresholds unchanged: email short<100, medium<400, long≥400).

In [1]:
from __future__ import annotations

import codecs
import mailbox
import re
import sys
import unicodedata
from pathlib import Path

import pandas as pd
from langdetect import detect, LangDetectException, DetectorFactory

DetectorFactory.seed = 0  # reproducibility

BASE     = Path('/Users/askar/projects/antifraud-deepfake-detection/v2')
sys.path.insert(0, str(BASE))
from config import length_bin as compute_length_bin  # channel-level thresholds from v2/config.py
RAW_PATH = BASE / 'data/raw/419_nigerian.txt'
OUT_DIR  = BASE / 'data/interim/gathered'
OUT_PATH = OUT_DIR / 'nigerian_fraud_prepared.jsonl'

URL_RE      = re.compile(r'(?i)(?:https?://\S+|www\.\S+|\b[a-z0-9][a-z0-9-]*\.[a-z]{2,}(?:/\S*)?)')
HTML_TAG_RE = re.compile(r'<[^>]+')
TOKEN_RE    = re.compile(r'\w+|[^\w\s]', re.UNICODE)

# Map non-standard or Windows charset strings to valid Python codec names
CHARSET_ALIASES: dict[str, str] = {
    'ansi':      'windows-1252',
    'windows':   'windows-1252',
    'x-unknown': 'latin-1',
    'unknown':   'latin-1',
    'default':   'latin-1',
    'us-ascii':  'ascii',
}


def safe_decode(payload: bytes, charset: str | None) -> str:
    charset = (charset or 'latin-1').strip().lower()
    charset = CHARSET_ALIASES.get(charset, charset)
    try:
        codecs.lookup(charset)
    except LookupError:
        charset = 'latin-1'
    return payload.decode(charset, errors='replace')


def normalize_text(text: str) -> str:
    text = '' if text is None else str(text)
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\uFFFD', ' ')
    text = re.sub(r'[\x00-\x1F\x7F]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def mask_url(text: str) -> str:
    # First pass: catch full URLs (http://domain/path, www.domain, bare domain)
    text = URL_RE.sub('[URL]', text)
    # Second pass: catch bare http(s):// fragments left when URL was space-broken
    text = re.sub(r'https?://\S*', '[URL]', text)
    return text


def strip_html(text: str) -> str:
    return HTML_TAG_RE.sub(' ', text)


def is_english(text: str) -> bool:
    """Detect English via langdetect (seed=0 for reproducibility)."""
    if not text or len(text.strip()) < 15:
        return False
    try:
        return detect(text[:2000]) == 'en'
    except LangDetectException:
        return False


def token_length(text: str) -> int:
    return len(TOKEN_RE.findall(text))


def extract_body(msg: mailbox.mboxMessage) -> str:
    """Extract text body from email message, preferring text/plain over text/html."""
    if msg.is_multipart():
        plain_parts: list[str] = []
        html_parts:  list[str] = []
        for part in msg.walk():
            ct = part.get_content_type()
            if ct == 'text/plain':
                payload = part.get_payload(decode=True)
                if payload:
                    plain_parts.append(safe_decode(payload, part.get_content_charset()))
            elif ct == 'text/html':
                payload = part.get_payload(decode=True)
                if payload:
                    html_parts.append(safe_decode(payload, part.get_content_charset()))
        if plain_parts:
            return '\n'.join(plain_parts)
        if html_parts:
            return strip_html('\n'.join(html_parts))
        return ''
    else:
        ct = msg.get_content_type()
        payload = msg.get_payload(decode=True)
        if not payload:
            payload_str = msg.get_payload()
            return payload_str if isinstance(payload_str, str) else ''
        text = safe_decode(payload, msg.get_content_charset())
        if ct == 'text/html':
            text = strip_html(text)
        return text

In [2]:
# Parse all messages from the mbox file
records: list[dict] = []
mbox_obj = mailbox.mbox(str(RAW_PATH))

for msg in mbox_obj:
    raw_body = extract_body(msg)
    records.append({
        'raw_body':  raw_body,
        'subject':   normalize_text(str(msg.get('Subject', '') or '')),
        'from_addr': str(msg.get('From', '') or ''),
        'date_raw':  str(msg.get('Date', '') or ''),
    })

print(f'Total messages parsed: {len(records)}')
print(f'Messages with empty body: {sum(1 for r in records if not r["raw_body"].strip())}')

Total messages parsed: 3978
Messages with empty body: 1


In [3]:
# Clean, filter, and deduplicate
df = pd.DataFrame(records)

df['text_norm']   = df['raw_body'].map(normalize_text)
df = df[df['text_norm'] != ''].copy()

# URL-masked text = final output text field
df['text_masked'] = df['text_norm'].map(mask_url).map(normalize_text)
df = df[df['text_masked'] != ''].copy()

df['is_english'] = df['text_masked'].map(is_english)
df_en = df[df['is_english']].copy()

# Quality gate: minimum character length after masking
df_en = df_en[df_en['text_masked'].str.len() >= 50].copy()

# Outlier cap: remove parsing artifacts (max observed ~26k tokens vs p90=758).
# Legitimate 419 emails do not exceed 5 000 tokens.
df_en = df_en[df_en['text_masked'].map(token_length) <= 5000].copy()

# Deduplicate by masked text (keep first occurrence, track duplicate count)
grp      = df_en.groupby('text_masked', sort=False)
prepared = grp.first().reset_index(drop=False)
prepared['n_duplicate_rows'] = grp.size().values

print(f'After text cleaning:  {len(df)}')
print(f'After english filter: {len(df_en)}')
print(f'After dedup:          {len(prepared)}')

After text cleaning:  3977
After english filter: 3934
After dedup:          3234


In [4]:
# Assign Core schema fields and write output
prepared['text']         = prepared['text_masked']
prepared['char_length']  = prepared['text'].str.len()
prepared['token_length'] = prepared['text'].map(token_length)
prepared['length_bin']   = prepared['token_length'].map(lambda t: compute_length_bin(t, 'email'))

# has_url based on pre-masked text (whether original had any URL)
prepared['has_url'] = prepared['text_norm'].str.contains(URL_RE).map(
    lambda v: 'yes' if v else 'no'
)

prepared['label']            = 0
prepared['label_str']        = 'human'
prepared['fraudness']        = 'fraud'
prepared['channel']          = 'email'
prepared['scenario_family']  = 'advance_fee_scam_email'
prepared['source_family']    = 'nigerian_fraud'
prepared['dataset_source']   = '419_nigerian.txt'
prepared['time_band']        = 'legacy'
prepared['origin_model']     = 'human'
prepared['split']            = 'tbd'
prepared['lang_filter_method'] = 'english_like_ascii_v1'

out_cols = [
    'text', 'label', 'label_str', 'fraudness', 'channel', 'scenario_family',
    'source_family', 'dataset_source', 'time_band', 'length_bin',
    'char_length', 'token_length', 'has_url',
    'subject', 'from_addr', 'date_raw',
    'origin_model', 'split', 'lang_filter_method', 'n_duplicate_rows',
]
out_df = prepared[out_cols].copy()

OUT_DIR.mkdir(parents=True, exist_ok=True)
out_df.to_json(OUT_PATH, orient='records', lines=True, force_ascii=False)

print(f'Written rows:         {len(out_df)}')
print(f'Output:               {OUT_PATH}')

print('\nlength_bin distribution:')
print(out_df['length_bin'].value_counts())

print('\nhas_url distribution:')
print(out_df['has_url'].value_counts())

Written rows:         3234
Output:               /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/gathered/nigerian_fraud_prepared.jsonl

length_bin distribution:
length_bin
long      2303
medium     903
short       28
Name: count, dtype: int64

has_url distribution:
has_url
yes    2912
no      322
Name: count, dtype: int64


In [5]:
# Validation and sanity checks
check = pd.read_json(OUT_PATH, lines=True)

assert check['text'].astype(str).str.strip().ne('').all(), 'empty texts found'
assert (check['label'] == 0).all(), 'unexpected label values'
assert (check['channel'] == 'email').all(), 'unexpected channel'
assert (check['scenario_family'] == 'advance_fee_scam_email').all(), 'unexpected scenario_family'
assert (check['fraudness'] == 'fraud').all(), 'unexpected fraudness'
assert (check['time_band'] == 'legacy').all(), 'unexpected time_band'
assert set(check['length_bin']).issubset({'short', 'medium', 'long'}), 'unexpected length_bin values'
assert check['text'].str.contains(r'https?://', regex=True).sum() == 0, \
    'some texts still contain unmasked URLs'

print('Validation passed.')
print(f'Rows in gathered JSONL: {len(check)}')
print(f'Columns: {list(check.columns)}')

print('\nSample texts (first 3):')
for t in check['text'].head(3):
    print(' ', t[:250])
    print()

print('Files in gathered/:')
for p in sorted(OUT_DIR.glob('*')):
    print(f'  - {p.name}')

Validation passed.
Rows in gathered JSONL: 3234
Columns: ['text', 'label', 'label_str', 'fraudness', 'channel', 'scenario_family', 'source_family', 'dataset_source', 'time_band', 'length_bin', 'char_length', 'token_length', 'has_url', 'subject', 'from_addr', 'date_raw', 'origin_model', 'split', 'lang_filter_method', 'n_duplicate_rows']

Sample texts (first 3):
  FROM:MR. JAMES NGOLA. CONFIDENTIAL TEL: 233-27-587908. E-MAIL: (james_ngola2002@[URL]). URGENT BUSINESS ASSISTANCE AND PARTNERSHIP. DEAR FRIEND, I AM ( DR.) JAMES NGOLA, THE PERSONAL ASSISTANCE TO THE LATE CONGOLESE (PRESIDENT LAURENT KABILA) WHO WAS

  Dear Friend, I am Mr. Ben Suleman a custom officer and work as Assistant controller of the Customs and Excise department Of the Federal Ministry of Internal Affairs stationed at the Murtala Mohammed International Airport, Ikeja, Lagos-Nigeria. After 

  FROM HIS ROYAL MAJESTY (HRM) CROWN RULER OF ELEME KINGDOM CHIEF DANIEL ELEME, PHD, EZE 1 OF ELEME.E-MAIL ADDRESS:obong_715@[URL